# Cascade validation

Бинарная проверка каскада на YOLO-разметке:

- `ACCEPT` + в GT есть bbox = `TP`
- `ACCEPT` + GT пустой = `FP`
- `REJECT` + GT пустой = `TN`
- `REJECT` + в GT есть bbox = `FN`


In [ ]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter

PROJECT_ROOT = Path(r"C:\MY_Data\Programming\Work\UGNTU_TGBOT")
ML_SERVICE_ROOT = PROJECT_ROOT / "ml_service"
DATASET_ROOT = Path(r"C:\MY_Data\Programming\Work\fashion\data\processed\detection")

SPLIT = "val"  # train или val
MAX_IMAGES = None  # например 50 для быстрой проверки

GARMENT_MODEL_PATH = ML_SERVICE_ROOT / "models" / "detector.onnx"
BAD_CLASSES_DETECTOR_PATH = ML_SERVICE_ROOT / "models" / "bad_classes_detector.onnx"

GARMENT_CONF = 0.25
BAD_CLASS_CONF = 0.35
PERSON_CONF = 0.75

assert GARMENT_MODEL_PATH.exists(), GARMENT_MODEL_PATH
assert BAD_CLASSES_DETECTOR_PATH.exists(), BAD_CLASSES_DETECTOR_PATH
assert (DATASET_ROOT / "images" / SPLIT).exists(), DATASET_ROOT / "images" / SPLIT
assert (DATASET_ROOT / "labels" / SPLIT).exists(), DATASET_ROOT / "labels" / SPLIT

sys.path.insert(0, str(ML_SERVICE_ROOT))


In [ ]:
from app.config import Settings
from app.pipeline import DetectionPipeline

settings = Settings(
    garment_model_path=str(GARMENT_MODEL_PATH),
    bad_classes_detector_path=str(BAD_CLASSES_DETECTOR_PATH),
    yolo_input_size=960,
    yolo_iou_threshold=0.45,
    garment_conf=GARMENT_CONF,
    bad_class_conf=BAD_CLASS_CONF,
    person_conf=PERSON_CONF,
    save_inference_results=False,
)

pipeline = DetectionPipeline.from_settings(settings)
print("pipeline loaded")


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_paths(split: str):
    root = DATASET_ROOT / "images" / split
    paths = sorted(p for p in root.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    return paths[:MAX_IMAGES] if MAX_IMAGES else paths

def label_path_for(image_path: Path) -> Path:
    return DATASET_ROOT / "labels" / SPLIT / f"{image_path.stem}.txt"

def has_gt_bbox(image_path: Path) -> bool:
    label_path = label_path_for(image_path)
    if not label_path.exists():
        return False
    return bool(label_path.read_text(encoding="utf-8").strip())

def confusion_bucket(decision: str, has_gt: bool) -> str:
    accepted = decision == "ACCEPT"
    if accepted and has_gt:
        return "TP"
    if accepted and not has_gt:
        return "FP"
    if not accepted and not has_gt:
        return "TN"
    return "FN"

paths = image_paths(SPLIT)
print(f"images: {len(paths)}")
print(f"gt positives: {sum(has_gt_bbox(p) for p in paths)}")
print(f"gt negatives: {sum(not has_gt_bbox(p) for p in paths)}")


In [ ]:
rows = []
counts = Counter()
reason_counts = Counter()

for index, image_path in enumerate(paths, start=1):
    result = pipeline.cascade(image_path.read_bytes())
    has_gt = has_gt_bbox(image_path)
    bucket = confusion_bucket(result["decision"], has_gt)

    counts[bucket] += 1
    reason_counts[result["reason"]] += 1
    rows.append({
        "image": image_path.name,
        "label": label_path_for(image_path).name,
        "has_gt_bbox": has_gt,
        "decision": result["decision"],
        "reason": result["reason"],
        "bucket": bucket,
        "garment_count": len(result["garment_detections"]),
        "bad_class_count": len(result["bad_class_detections"]),
        "bad_classes": ",".join(sorted({d.get("class_name", "") for d in result["bad_class_detections"]})),
    })

    if index % 25 == 0 or index == len(paths):
        print(f"{index}/{len(paths)}", dict(counts))

print("done")


In [ ]:
tp = counts["TP"]
tn = counts["TN"]
fp = counts["FP"]
fn = counts["FN"]
total = tp + tn + fp + fn

def safe_div(num, den):
    return num / den if den else 0.0

metrics = {
    "total": total,
    "TP": tp,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "accuracy": safe_div(tp + tn, total),
    "precision": safe_div(tp, tp + fp),
    "recall": safe_div(tp, tp + fn),
    "specificity": safe_div(tn, tn + fp),
    "f1": safe_div(2 * tp, 2 * tp + fp + fn),
}

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("reasons:", dict(reason_counts))


In [ ]:
out_csv = ML_SERVICE_ROOT / f"cascade_validation_{SPLIT}.csv"
with out_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else [])
    if rows:
        writer.writeheader()
        writer.writerows(rows)

print(out_csv)


In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame(rows).head(20))
    display(pd.DataFrame(rows).groupby(["bucket", "reason"]).size().reset_index(name="count"))
except ImportError:
    rows[:5]
